# [센서 실습 11] 미세먼지 센서 – PMS7003

레이저 산란식 미세먼지 센서 **PMS7003**이 UART로 보내는 **32바이트 데이터 프레임**을 직접 해석하여 PM2.5·PM10 농도를 구하고, 환경부 기준으로 등급을 매깁니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
PMS7003 안에서 작은 팬이 공기를 빨아들이고, **레이저**를 쏘아 먼지 입자에 부딪혀 흩어지는 빛(**산란광**)을 측정합니다.
- 입자가 클수록 산란광이 강함 → 입자 크기별 개수를 세어 질량 농도(㎍/㎥)로 계산
- 약 1초마다 **32바이트 프레임**을 UART(9600bps)로 보냄

| 바이트 | 내용 |
|---|---|
| 0~1 | 시작 문자 `0x42 0x4D` ('B', 'M') |
| 2~3 | 프레임 길이 (28) |
| 4~9 | PM1.0 / PM2.5 / PM10 (공장 표준 CF=1) |
| **10~15** | **PM1.0 / PM2.5 / PM10 (대기 환경 기준)** ← 사용 |
| 16~27 | 0.1L 안의 크기별 입자 개수 |
| 30~31 | 체크섬 (0~29번 바이트의 합) |

값은 모두 2바이트(상위·하위)로 되어 있어 `상위 × 256 + 하위`로 계산합니다.

### 부품 사양
| 항목 | 내용 |
|---|---|
| 측정 대상 | PM1.0, PM2.5, PM10 |
| 측정 범위 | 0 ~ 500 ㎍/㎥ (유효) |
| 오차 | ±10 ㎍/㎥ (0~100), ±10% (100~500) |
| 응답 | 약 1초 간격 출력 |
| 전원 | 5V (팬 구동), 신호 3.3V |
| 통신 | UART 9600bps |

## 2. 라즈베리파이 연결 방법
**준비물**: PMS7003 + 어댑터 보드(1.27mm 커넥터 변환), 점퍼선

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| PMS7003 VCC | 5V (2번 핀) |  |
| PMS7003 GND | GND (6번 핀) |  |
| PMS7003 TX | GPIO15 / RXD (10번 핀) | 센서 송신 → Pi 수신 |
| PMS7003 RX | GPIO14 / TXD (8번 핀) | Pi 송신 → 센서 수신 |

PMS7003의 신호선은 3.3V 레벨이므로 라즈베리파이와 바로 연결할 수 있습니다. 전원만 5V가 필요합니다.

**인터페이스 활성화**
- **UART**: `sudo raspi-config` → Interface Options → **Serial Port** → 로그인 셸: 아니요, 시리얼 하드웨어: 예

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install pyserial
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
32바이트 프레임을 검사(시작 문자·길이·체크섬)하고 PM 값을 꺼냅니다. 체크섬이 틀리면 `None`을 돌려줍니다.

In [ ]:
def parse_pms7003(frame):
    """PMS7003 32바이트 프레임 → {'pm1_0', 'pm2_5', 'pm10'} (대기 환경 기준, ㎍/㎥)"""
    if frame is None or len(frame) != 32 or frame[0] != 0x42 or frame[1] != 0x4D:
        return None
    if sum(frame[:30]) & 0xFFFF != (frame[30] << 8 | frame[31]):   # 체크섬 검사
        return None
    word = lambda i: frame[i] << 8 | frame[i + 1]
    return {"pm1_0": word(10), "pm2_5": word(12), "pm10": word(14)}

def pm25_grade(v):
    """환경부 PM2.5 예보 등급 (㎍/㎥)"""
    return "좋음" if v <= 15 else "보통" if v <= 35 else "나쁨" if v <= 75 else "매우나쁨"

def pm10_grade(v):
    return "좋음" if v <= 30 else "보통" if v <= 80 else "나쁨" if v <= 150 else "매우나쁨"

### 4-2. 실제 센서 클래스
시리얼 포트에서 시작 문자 `0x42 0x4D`를 찾은 뒤 나머지 30바이트를 읽어 프레임을 완성합니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealPMS7003:
    def __init__(self, port="/dev/serial0"):
        import serial
        self.ser = serial.Serial(port, baudrate=9600, timeout=2)

    def read(self, tries=64):
        for _ in range(tries):
            if self.ser.read(1) == b"\x42" and self.ser.read(1) == b"\x4d":
                frame = b"\x42\x4d" + self.ser.read(30)
                data = parse_pms7003(frame)
                if data:
                    return data
        return {"pm1_0": None, "pm2_5": None, "pm10": None}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 상황 | 베란다에 설치, 48시간 1시간 간격 | 실외 공기 |
| 평상시 PM2.5 | 15 ~ 30 ㎍/㎥, 출퇴근 시간대 상승 | 도시의 일반적인 농도 |
| 고농도 사례 | 둘째 날 오후 국외 유입으로 최대 약 90 ㎍/㎥ | 봄철 고농도 사례 |
| PM 비율 | PM1.0 ≈ PM2.5 × 0.7, PM10 ≈ PM2.5 × 1.6 | 일반적인 도시 대기 비율 |
| 통신 오류 | 2% 확률로 체크섬 오류 | 실제 UART 수신 오류 재현 |

가상 센서는 **실제 프로토콜과 같은 32바이트 프레임(체크섬 포함)** 을 만들어 해석기에 넘깁니다.

In [ ]:
class VirtualPMS7003:
    def __init__(self, dt=3600, seed=11):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    @staticmethod
    def make_frame(pm1, pm25, pm10):
        """실제 센서와 같은 32바이트 프레임 만들기"""
        words = [28, pm1, pm25, pm10, pm1, pm25, pm10] + [0] * 6 + [0]   # 길이, CF1, 대기, 입자수, 예약
        body = [0x42, 0x4D]
        for w in words:
            body += [w >> 8, w & 0xFF]
        cs = sum(body) & 0xFFFF
        return bytes(body + [cs >> 8, cs & 0xFF])

    def read(self):
        h = self.t / 3600
        pm25 = 20 + 6 * np.exp(-((h % 24 - 8) / 2) ** 2) + 6 * np.exp(-((h % 24 - 19) / 2) ** 2)  # 출퇴근
        pm25 += 70 * np.exp(-((h - 38) / 5) ** 2)                                                  # 국외 유입
        pm25 = max(1, pm25 + self.rng.normal(0, 3))
        frame = bytearray(self.make_frame(int(pm25 * 0.7), int(pm25), int(pm25 * 1.6)))
        if self.rng.random() < 0.02:
            frame[20] ^= 0xFF                                     # 수신 중 비트 오류
        self.t += self.dt
        data = parse_pms7003(bytes(frame))
        return data if data else {"pm1_0": None, "pm2_5": None, "pm10": None}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualPMS7003(dt=3600)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
1시간 간격으로 48시간 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 48               # 측정 횟수
INTERVAL = 3600               # 측정 간격(초)

sensor = RealPMS7003() if USE_REAL_SENSOR else VirtualPMS7003(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
환경부 미세먼지 예보 등급으로 판정합니다.

| 등급 | PM2.5 (초미세먼지) | PM10 (미세먼지) |
|---|---|---|
| 좋음 | 0 ~ 15 | 0 ~ 30 |
| 보통 | 16 ~ 35 | 31 ~ 80 |
| 나쁨 | 36 ~ 75 | 81 ~ 150 |
| 매우나쁨 | 76 이상 | 151 이상 |

In [ ]:
df["hour"] = df["time_s"] / 3600
print("체크섬 오류로 버린 프레임:", df["pm2_5"].isna().sum(), "개")
df[["pm1_0", "pm2_5", "pm10"]] = df[["pm1_0", "pm2_5", "pm10"]].interpolate().bfill()
df["grade_pm25"] = df["pm2_5"].apply(pm25_grade)
df["grade_pm10"] = df["pm10"].apply(pm10_grade)
print("\nPM2.5 등급별 시간:"); print(df["grade_pm25"].value_counts().to_string())
bad = df[df["grade_pm25"].isin(["나쁨", "매우나쁨"])]
if len(bad):
    print(f"\n⚠️ 나쁨 이상: {bad['hour'].min():.0f} ~ {bad['hour'].max():.0f}시간째 → 외출 자제, 공기청정기 가동")
print(f"하루 평균 PM2.5: 1일차 {df[df.hour < 24]['pm2_5'].mean():.1f}, 2일차 {df[df.hour >= 24]['pm2_5'].mean():.1f} ㎍/㎥")

## 8. 시각화

In [ ]:
plt.figure(figsize=(10, 4.5))
for lo, hi, name, g in [(0, 15, "좋음", "0.97"), (15, 35, "보통", "0.90"), (35, 75, "나쁨", "0.80"), (75, 130, "매우나쁨", "0.68")]:
    plt.axhspan(lo, hi, color=g); plt.text(48.5, (lo + hi) / 2, name, va="center")
plt.plot(df["hour"], df["pm2_5"], "k-o", ms=3, lw=1.5, label="PM2.5")
plt.plot(df["hour"], df["pm10"], "k--", lw=1, label="PM10")
plt.ylim(0, 130); plt.xlim(0, 48)
plt.title("PMS7003 미세먼지 측정 (48시간)"); plt.xlabel("경과 시간 (시간)"); plt.ylabel("㎍/㎥")
plt.legend(loc="upper left"); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "미세먼지센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "PMS7003",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/dust", payload)
# client.disconnect()

## 직접 해 보기
- 가상 센서가 만든 프레임 하나를 `print(list(VirtualPMS7003.make_frame(10, 20, 30)))`로 찍어 보고 바이트 위치를 확인해 보세요.
- PM2.5가 '나쁨'이면 공기청정기 릴레이를 켜는 열(`purifier`)을 추가해 보세요.
- PM10 등급으로도 그래프를 그려 PM2.5 등급과 비교해 보세요.